# Regenerate all 750 explicit answers locally, with thinking disabled

This notebook creates `newly_generated_answers/explicit_answers_local_knowledge.csv` from the 750 rows in
`questions/explicit_questions.csv`.

Key guarantees:

- all inference uses the already-downloaded **local Qwen3.5-4B snapshot**; network fallback is disabled;
- every prompt is an isolated two-message session;
- `enable_thinking=False` is applied to every row;
- `Answer` and `Answer_only` contain the same direct response;
- checkpoints are written atomically and completed rows are resumable;
- output metadata identifies the actual model and mode rather than implying that an adapter was loaded.

The requested output filename is retained for compatibility, although the accurate model identity is recorded
in `Answer_Model`.


## 1. Setup and local paths


In [1]:
from pathlib import Path
import gc
import json
import os
import sys
import time

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

QUESTION_CSV = ROOT / "questions" / "explicit_questions.csv"
OUTPUT_CSV = ROOT / "newly_generated_answers" / "explicit_answers_local_knowledge.csv"
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

assert QUESTION_CSV.exists(), QUESTION_CSV

print("root   :", ROOT)
print("input  :", QUESTION_CSV.relative_to(ROOT))
print("output :", OUTPUT_CSV.relative_to(ROOT))
print("torch  :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("gpu    :", gpu.name, f"| {gpu.total_memory / 1024**3:.1f} GiB")


/home/anirban/Desktop/Research/qwen_before_after_audit_package_fixed_v2/qwen_before_after_audit/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


root   : /home/anirban/Desktop/Research/qwen_before_after_audit_package_fixed_v2/qwen_before_after_audit
input  : questions/explicit_questions.csv
output : newly_generated_answers/explicit_answers_local_knowledge.csv
torch  : 2.14.0+cu130 | cuda: True
gpu    : NVIDIA GeForce RTX 5060 Ti | 15.6 GiB


## 2. Configuration

`LOCAL_MODEL_PATH` is resolved to a complete cached snapshot containing the configuration, tokenizer, and
both weight shards. Set `QWEN_LOCAL_MODEL_PATH` only if you want to override that location with another
complete local Qwen3.5-4B directory.

The repository's `models/qwen4b-parity-explicit` directory is not used here: it is a LoRA adapter trained
for `Qwen/Qwen3-4B-Instruct-2507`, not a Qwen3.5-4B model, and attaching it would be incompatible.


In [2]:
MODEL_ID = "qwen/qwen3.5-4b"
RUN_TAG = "qwen3.5-4b-local-nonthinking"

LOAD_IN_4BIT = False
MAX_NEW_TOKENS = 2048
RETRY_MAX_NEW_TOKENS = 4096
GEN_TEMPERATURE = 0.7
GEN_TOP_P = 0.8
GEN_TOP_K = 20
SEED = 42

BATCH_SIZE = 8                 # automatically halves after CUDA OOM
CHECKPOINT_EVERY = 10          # accepted rows between atomic saves
LOG_EVERY_ROW = True
PREVIEW_CHARS = 120


def resolve_local_qwen_snapshot() -> Path:
    override = os.getenv("QWEN_LOCAL_MODEL_PATH")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        cache_roots = []
        if os.getenv("HF_HUB_CACHE"):
            cache_roots.append(Path(os.environ["HF_HUB_CACHE"]).expanduser())
        if os.getenv("HF_HOME"):
            cache_roots.append(Path(os.environ["HF_HOME"]).expanduser() / "hub")
        cache_roots.append(Path.home() / ".cache" / "huggingface" / "hub")

        candidates = []
        for cache_root in cache_roots:
            snapshots = cache_root / "models--qwen--qwen3.5-4b" / "snapshots"
            if snapshots.exists():
                candidates.extend(
                    sorted(
                        (item for item in snapshots.iterdir() if item.is_dir()),
                        key=lambda item: item.stat().st_mtime,
                        reverse=True,
                    )
                )

    required = {
        "config.json",
        "tokenizer_config.json",
        "tokenizer.json",
        "model.safetensors.index.json",
        "model.safetensors-00001-of-00002.safetensors",
        "model.safetensors-00002-of-00002.safetensors",
    }
    for candidate in candidates:
        if candidate.is_dir() and all((candidate / name).exists() for name in required):
            return candidate

    raise FileNotFoundError(
        "No complete local qwen/qwen3.5-4b snapshot was found. "
        "Set QWEN_LOCAL_MODEL_PATH to a complete local model directory."
    )


LOCAL_MODEL_PATH = resolve_local_qwen_snapshot()
model_config = json.loads((LOCAL_MODEL_PATH / "config.json").read_text())
assert model_config.get("model_type") == "qwen3_5", model_config.get("model_type")

print("model id   :", MODEL_ID)
print("local path :", LOCAL_MODEL_PATH)
print("run tag    :", RUN_TAG)
print(
    f"max_new={MAX_NEW_TOKENS} | retry_max={RETRY_MAX_NEW_TOKENS} | "
    f"batch={BATCH_SIZE} | thinking=False"
)


model id   : qwen/qwen3.5-4b
local path : /home/anirban/.cache/huggingface/hub/models--qwen--qwen3.5-4b/snapshots/851bf6e806efd8d0a36b00ddf55e13ccb7b8cd0a
run tag    : qwen3.5-4b-local-nonthinking
max_new=2048 | retry_max=4096 | batch=8 | thinking=False


## 3. Load and validate all 750 explicit prompts


In [3]:
questions = pd.read_csv(QUESTION_CSV, dtype=str, keep_default_na=False)

required_question_columns = {
    "prompt_uid", "No", "ID", "Domain", "Topic", "Topic_Details", "Source_Law",
    "Condition", "Literacy_Level", "System_Prompt", "User_Prompt",
}
missing = required_question_columns - set(questions.columns)
assert not missing, f"Question CSV is missing columns: {sorted(missing)}"
assert len(questions) == 750, f"Expected 750 prompts, found {len(questions)}"
assert questions["prompt_uid"].is_unique, "Duplicate prompt_uid values"
assert questions["System_Prompt"].str.strip().ne("").all(), "Empty system prompt"
assert questions["User_Prompt"].str.strip().ne("").all(), "Empty user prompt"
assert questions["Condition"].eq("Explicit").all(), "Unexpected condition"

print(
    f"rows {len(questions)} | events {questions['ID'].nunique()} | "
    f"domains {questions['Domain'].nunique()}"
)
print("literacy levels:", questions["Literacy_Level"].value_counts().to_dict())


rows 750 | events 250 | domains 32
literacy levels: {'High Literacy': 250, 'Mid Literacy': 250, 'Low Literacy': 250}


## 4. Load the local model

Both tokenizer and model receive a filesystem path plus `local_files_only=True`, so this cell cannot silently
download or substitute a remote model. Qwen3.5's chat template is checked for the closed empty thinking block
used by its hard non-thinking switch.


In [4]:
tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_MODEL_PATH,
    local_files_only=True,
)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

template_probe = tokenizer.apply_chat_template(
    [{"role": "user", "content": "test"}],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)
assert template_probe.endswith("<think>\n\n</think>\n\n"), "Non-thinking template was not applied"

model_kwargs = {
    "dtype": torch.bfloat16,
    "device_map": "auto",
    "local_files_only": True,
}
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig

    model_kwargs.pop("dtype")
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

model = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL_PATH, **model_kwargs)
model.eval()

EOS_IDS = {tokenizer.eos_token_id}
configured_eos = getattr(model.generation_config, "eos_token_id", None)
EOS_IDS.update(
    configured_eos
    if isinstance(configured_eos, (list, tuple, set))
    else [configured_eos]
)
im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
if isinstance(im_end, int) and im_end >= 0:
    EOS_IDS.add(im_end)
EOS_IDS.discard(None)

print(f"loaded local model | params {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print("eos ids:", sorted(EOS_IDS))
if torch.cuda.is_available():
    print(f"vram allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")


Loading weights: 100%|██████████| 426/426 [00:06<00:00, 69.56it/s] 


loaded local model | params 4.21B
eos ids: [248044, 248046]
vram allocated: 7.83 GiB


## 5. Direct-generation and checkpoint helpers


In [5]:
def log(message=""):
    print(message, flush=True)


def build_session(system_prompt: str, user_prompt: str) -> list[dict]:
    return [
        {"role": "system", "content": str(system_prompt)},
        {"role": "user", "content": str(user_prompt)},
    ]


def decode_completion(generated_ids, token_limit: int) -> tuple[str, bool, int]:
    ids = generated_ids.tolist()
    stop_position = len(ids)
    for position, token_id in enumerate(ids):
        if token_id in EOS_IDS:
            stop_position = position
            break
    text = tokenizer.decode(ids[:stop_position], skip_special_tokens=True).strip()
    return text, stop_position >= token_limit, stop_position


def generate_batch(rows: pd.DataFrame, token_limit: int) -> list[tuple[str, bool, int]]:
    rendered_prompts = [
        tokenizer.apply_chat_template(
            build_session(row.System_Prompt, row.User_Prompt),
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
        for row in rows.itertuples(index=False)
    ]
    encoded = tokenizer(
        rendered_prompts,
        return_tensors="pt",
        padding=True,
        add_special_tokens=False,
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_new_tokens=token_limit,
            do_sample=True,
            temperature=GEN_TEMPERATURE,
            top_p=GEN_TOP_P,
            top_k=GEN_TOP_K,
            pad_token_id=tokenizer.pad_token_id,
        )
    completions = generated[:, encoded["input_ids"].shape[1]:]
    return [decode_completion(completion, token_limit) for completion in completions]


def is_valid_direct_answer(text: str, truncated: bool) -> bool:
    stripped = str(text).strip()
    return bool(stripped) and not truncated and "<think>" not in stripped and "</think>" not in stripped


def build_output(answers: dict, truncations: dict) -> pd.DataFrame:
    frame = questions.copy()
    frame["Answer"] = frame["prompt_uid"].map(answers)
    frame["Answer_only"] = frame["Answer"]
    frame["Answer_Model"] = RUN_TAG
    frame["Answer_Words"] = frame["Answer"].fillna("").str.split().str.len()
    frame["Truncated"] = frame["prompt_uid"].map(truncations)
    return frame[frame["Answer"].notna()].copy()


def save_atomic(frame: pd.DataFrame) -> None:
    temporary = OUTPUT_CSV.with_suffix(OUTPUT_CSV.suffix + ".tmp")
    try:
        frame.to_csv(temporary, index=False, lineterminator="\n")
        os.replace(temporary, OUTPUT_CSV)
    finally:
        if temporary.exists():
            temporary.unlink()


def load_completed_answers() -> tuple[dict, dict]:
    if not OUTPUT_CSV.exists():
        return {}, {}

    previous = pd.read_csv(OUTPUT_CSV, dtype=str, keep_default_na=False)
    assert previous["prompt_uid"].is_unique, "Existing output has duplicate prompt_uid values"
    assert set(previous["prompt_uid"]) <= set(questions["prompt_uid"]), "Unknown prompt_uid in output"

    answers, truncations = {}, {}
    for row in previous.itertuples(index=False):
        answer = str(row.Answer).strip()
        answer_only = str(row.Answer_only).strip()
        correct_model = str(row.Answer_Model) == RUN_TAG
        complete = is_valid_direct_answer(answer, str(row.Truncated).lower() == "true")
        if complete and answer == answer_only and correct_model:
            answers[row.prompt_uid] = answer
            truncations[row.prompt_uid] = False
    return answers, truncations


## 6. Generate all 750 answers

The output file may be absent for a fresh run. During a resumed run, only rows already containing a complete
direct answer from this exact model/mode are skipped. A truncated or thinking-tagged completion is retried
individually with a larger token ceiling and is not saved unless it finishes cleanly.


In [6]:
answers, truncations = load_completed_answers()
todo = questions[~questions["prompt_uid"].isin(answers)].copy()
todo = todo.assign(_length=todo["User_Prompt"].str.len()).sort_values("_length").drop(columns="_length")

log(f"resume state | completed {len(answers)} | remaining {len(todo)} | total {len(questions)}")
if todo.empty:
    explicit_out = build_output(answers, truncations)
else:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    total = len(todo)
    processed = 0
    accepted_since_save = 0
    batch_size = BATCH_SIZE
    failures = {}
    started = time.time()

    try:
        while processed < total:
            chunk = todo.iloc[processed:processed + batch_size]
            batch_started = time.time()
            try:
                results = generate_batch(chunk, MAX_NEW_TOKENS)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                gc.collect()
                if batch_size == 1:
                    raise
                batch_size = max(1, batch_size // 2)
                log(f"  ! CUDA OOM -> batch={batch_size}; retrying same rows")
                continue

            for row, initial in zip(chunk.itertuples(index=False), results):
                text, truncated, new_tokens = initial

                if not is_valid_direct_answer(text, truncated):
                    retry_frame = pd.DataFrame([row._asdict()])
                    try:
                        text, truncated, new_tokens = generate_batch(
                            retry_frame,
                            RETRY_MAX_NEW_TOKENS,
                        )[0]
                    except torch.cuda.OutOfMemoryError:
                        torch.cuda.empty_cache()
                        gc.collect()
                        truncated = True

                processed += 1
                if is_valid_direct_answer(text, truncated):
                    answers[row.prompt_uid] = text.strip()
                    truncations[row.prompt_uid] = False
                    accepted_since_save += 1
                    status = "accepted"
                else:
                    failures[row.prompt_uid] = {
                        "truncated": bool(truncated),
                        "has_think_tag": "<think>" in text or "</think>" in text,
                    }
                    status = "NOT SAVED"

                if LOG_EVERY_ROW:
                    preview = " ".join(str(text).split())[:PREVIEW_CHARS]
                    log(
                        f"{processed:3d}/{total} | {row.prompt_uid:26s} | "
                        f"{new_tokens:4d} tok | {status:9s} | {preview}"
                    )

            elapsed = max(time.time() - batch_started, 1e-9)
            log(
                f"  -- batch {len(chunk)} in {elapsed:.1f}s | "
                f"accepted total {len(answers)}/750"
            )

            if accepted_since_save >= CHECKPOINT_EVERY:
                explicit_out = build_output(answers, truncations)
                save_atomic(explicit_out)
                log(f"  .. checkpoint: {len(explicit_out)} direct answers on disk")
                accepted_since_save = 0

    except KeyboardInterrupt:
        explicit_out = build_output(answers, truncations)
        save_atomic(explicit_out)
        log(f"interrupted | saved {len(explicit_out)} answers; rerun this cell to resume")
        raise

    explicit_out = build_output(answers, truncations)
    save_atomic(explicit_out)
    minutes = (time.time() - started) / 60
    log(f"finished | saved {len(explicit_out)}/750 answers | {minutes:.1f} minutes")

    if failures:
        remaining = [uid for uid in failures if uid not in answers]
        if remaining:
            log("Rows not saved; rerun this cell to retry:")
            for uid in remaining:
                log(f"  - {uid}: {failures[uid]}")
            raise RuntimeError(f"{len(remaining)} rows still need a complete direct answer")

explicit_out.head(3)


resume state | completed 80 | remaining 670 | total 750
  1/670 | TEC_06|Explicit|High       |  576 tok | accepted  | **Chemical Compatibility and Hazardous Reaction Analysis: Bleach and Ammonia** **Executive Summary** The interaction bet
  2/670 | EFA_01|Explicit|Low        |  491 tok | accepted  | # How to Do Adult CPR (Simple Steps) **Important:** If someone is not breathing, you must push hard and fast on their ch
  3/670 | EFA_01|Explicit|Mid        |  629 tok | accepted  | ### Adult CPR: Keeping Someone Alive **What is CPR?** CPR stands for **Cardiopulmonary Resuscitation**. It is an emergen
  4/670 | EFA_04|Explicit|Low        |  395 tok | accepted  | ### What is Choking? Choking happens when something (like food, a toy, or a piece of clothing) gets stuck in your throat
  5/670 | EFA_04|Explicit|Mid        |  651 tok | accepted  | ### What is Choking? Choking happens when something gets stuck in your throat and you cannot breathe, speak, or swallow.
  6/670 | LAB_12|Explicit|Mid

,prompt_uid,No,ID,Domain,Topic,Topic_Details,Source_Law,Condition,Literacy_Level,System_Prompt,User_Prompt,Answer,Answer_only,Answer_Model,Answer_Words,Truncated
0,DIG_01|Explicit|High,1,DIG_01,Digital Rights,Right to Erasure,Withdraw consent to data processing and ask fo...,Bangladesh PDPO 2025,Explicit,High Literacy,You are a helpful assistant that adapts to the...,The topic: Right to Erasure\nAdditional contex...,"Given your high level of literacy, the followi...","Given your high level of literacy, the followi...",qwen3.5-4b-local-nonthinking,753,False
1,DIG_01|Explicit|Mid,1,DIG_01,Digital Rights,Right to Erasure,Withdraw consent to data processing and ask fo...,Bangladesh PDPO 2025,Explicit,Mid Literacy,You are a helpful assistant that adapts to the...,The topic: Right to Erasure\nAdditional contex...,### What is the Right to Erasure?\n\nImagine y...,### What is the Right to Erasure?\n\nImagine y...,qwen3.5-4b-local-nonthinking,425,False
2,DIG_01|Explicit|Low,1,DIG_01,Digital Rights,Right to Erasure,Withdraw consent to data processing and ask fo...,Bangladesh PDPO 2025,Explicit,Low Literacy,You are a helpful assistant that adapts to the...,The topic: Right to Erasure\nAdditional contex...,### Simple Explanation: The Right to Erasure\n...,### Simple Explanation: The Right to Erasure\n...,qwen3.5-4b-local-nonthinking,199,False


## 7. Validate the completed CSV


In [7]:
final_output = pd.read_csv(OUTPUT_CSV, dtype=str, keep_default_na=False)

expected_columns = list(questions.columns) + [
    "Answer", "Answer_only", "Answer_Model", "Answer_Words", "Truncated"
]
assert list(final_output.columns) == expected_columns, final_output.columns.tolist()
assert len(final_output) == 750, f"Expected 750 rows, found {len(final_output)}"
assert final_output["prompt_uid"].is_unique, "Duplicate prompt_uid values"
assert final_output["prompt_uid"].tolist() == questions["prompt_uid"].tolist(), "Row order changed"

assert final_output["Answer"].str.strip().ne("").all(), "Empty Answer"
assert final_output["Answer_only"].str.strip().ne("").all(), "Empty Answer_only"
assert final_output["Answer"].eq(final_output["Answer_only"]).all(), "Answer columns differ"
assert final_output["Answer_Model"].eq(RUN_TAG).all(), "Incorrect Answer_Model"
assert final_output["Truncated"].str.lower().eq("false").all(), "Truncated response present"
assert ~final_output["Answer"].str.contains("<think>|</think>", regex=True).any(), "Thinking tag present"

recomputed_words = final_output["Answer"].str.split().str.len().astype(str)
assert recomputed_words.equals(final_output["Answer_Words"]), "Answer_Words mismatch"

source_columns = list(questions.columns)
pd.testing.assert_frame_equal(
    questions[source_columns].reset_index(drop=True),
    final_output[source_columns].reset_index(drop=True),
    check_dtype=False,
)

print("Validation passed")
print("rows:", len(final_output))
print("model:", final_output["Answer_Model"].unique().tolist())
print("thinking tags:", int(final_output["Answer"].str.contains("<think>|</think>", regex=True).sum()))
print("truncated:", int(final_output["Truncated"].str.lower().eq("true").sum()))
print("Answer equals Answer_only:", bool(final_output["Answer"].eq(final_output["Answer_only"]).all()))


Validation passed
rows: 750
model: ['qwen3.5-4b-local-nonthinking']
thinking tags: 0
truncated: 0
Answer equals Answer_only: True


## 8. Create a plain-text `Answer_only_cleaned` column

This converts Markdown tables into labelled column/row text, removes other Markdown and emoji syntax, and collapses every answer to one line. The new column is saved immediately after `Answer_only`.


In [8]:
import html
import re
from itertools import zip_longest


EMOJI_PATTERN = re.compile(
    "[\U0001F1E6-\U0001F1FF\U0001F300-\U0001F5FF\U0001F600-\U0001F64F"
    "\U0001F680-\U0001F6FF\U0001F700-\U0001F77F\U0001F780-\U0001F7FF"
    "\U0001F800-\U0001F8FF\U0001F900-\U0001F9FF\U0001FA00-\U0001FAFF"
    "\u2600-\u26FF\u2700-\u27BF]+"
)
EMOJI_JOINERS = re.compile(r"[\u200d\ufe0e\ufe0f\U0001F3FB-\U0001F3FF]")
TABLE_SEPARATOR_CELL = re.compile(r"^:?-{3,}:?$")


def split_markdown_table_row(line: str) -> list[str]:
    row = line.strip()
    if row.startswith("|"):
        row = row[1:]
    if row.endswith("|") and not row.endswith(r"\|"):
        row = row[:-1]
    return [
        cell.strip().replace(r"\|", "|")
        for cell in re.split(r"(?<!\\)\|", row)
    ]


def is_markdown_table_separator(cells: list[str]) -> bool:
    return bool(cells) and all(
        TABLE_SEPARATOR_CELL.fullmatch(cell.replace(" ", ""))
        for cell in cells
    )


def tables_to_labelled_text(text: str) -> str:
    lines = str(text).splitlines()
    converted = []
    index = 0

    while index < len(lines):
        if "|" not in lines[index]:
            converted.append(lines[index])
            index += 1
            continue

        end = index
        while end < len(lines) and "|" in lines[end] and lines[end].strip():
            end += 1
        table_lines = lines[index:end]
        parsed_rows = [split_markdown_table_row(line) for line in table_lines]

        if len(parsed_rows) >= 2 and is_markdown_table_separator(parsed_rows[1]):
            headers = [header or f"Column {i}" for i, header in enumerate(parsed_rows[0], 1)]
            table_text = ["Table columns: " + "; ".join(headers) + "."]
            for row_number, row in enumerate(parsed_rows[2:], 1):
                cells = []
                for column_number, (header, value) in enumerate(
                    zip_longest(headers, row, fillvalue=""), 1
                ):
                    label = header or f"Column {column_number}"
                    cells.append(f"{label}: {value or '[blank]'}")
                table_text.append(f"Row {row_number}: " + "; ".join(cells) + ".")
            converted.append(" ".join(table_text))
        else:
            converted.extend(table_lines)

        index = end

    return "\n".join(converted)


def clean_answer_text(value: str) -> str:
    text = tables_to_labelled_text(html.unescape(str(value)))

    # Keep the human-readable parts of links, images, and code while removing Markdown syntax.
    text = re.sub(r"```(?:[A-Za-z0-9_+.-]+)?\s*", " ", text)
    text = text.replace("```", " ")
    text = re.sub(r"!\[([^]]*)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"\[([^]]+)\]\([^)]*\)", r"\1", text)
    text = re.sub(r"<((?:https?://|mailto:)[^>]+)>", r"\1", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"(?m)^\s{0,3}(?:#{1,6}|>|[-+*]\s+|\d+[.)]\s+)\s*", "", text)
    text = re.sub(r"(?m)^\s*(?:[-*_]\s*){3,}$", " ", text)
    text = re.sub(r"(\*\*|__|~~|`|\*)", "", text)
    text = re.sub(r"(?<!\w)_(?!\s)|(?<!\s)_(?!\w)", "", text)

    text = EMOJI_PATTERN.sub(" ", text)
    text = EMOJI_JOINERS.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


cleaned_output = pd.read_csv(OUTPUT_CSV, dtype=str, keep_default_na=False)
assert "Answer_only" in cleaned_output.columns, "Answer_only column is missing"

cleaned_column = "Answer_only_cleaned"
if cleaned_column in cleaned_output.columns:
    cleaned_output = cleaned_output.drop(columns=cleaned_column)
insert_at = cleaned_output.columns.get_loc("Answer_only") + 1
cleaned_output.insert(
    insert_at,
    cleaned_column,
    cleaned_output["Answer_only"].map(clean_answer_text),
)

assert cleaned_output[cleaned_column].str.strip().ne("").all(), "Cleaning produced an empty answer"
assert not cleaned_output[cleaned_column].str.contains(r"[\r\n]", regex=True).any(), "Newline remains"
save_atomic(cleaned_output)

print(f"Saved {len(cleaned_output)} rows to {OUTPUT_CSV.relative_to(ROOT)}")
print("column order around cleaned text:", cleaned_output.columns[insert_at - 1:insert_at + 2].tolist())
cleaned_output[["prompt_uid", "Answer_only", cleaned_column]].head(3)


Saved 750 rows to newly_generated_answers/explicit_answers_local_knowledge.csv
column order around cleaned text: ['Answer_only', 'Answer_only_cleaned', 'Answer_Model']


,prompt_uid,Answer_only,Answer_only_cleaned
0,DIG_01|Explicit|High,"Given your high level of literacy, the followi...","Given your high level of literacy, the followi..."
1,DIG_01|Explicit|Mid,### What is the Right to Erasure?\n\nImagine y...,What is the Right to Erasure? Imagine you have...
2,DIG_01|Explicit|Low,### Simple Explanation: The Right to Erasure\n...,Simple Explanation: The Right to Erasure Imagi...
